# Decision records and evidence-gated promotion

Every comparison in the [lifecycle](../docs/genai-lifecycle.md) ends in an
explicit decision — `adopt`, `reject`, or `inconclusive` — and only adopted
changes may move a production prompt alias. This credential-free lab walks the
complete loop with the shared SDK toolkit:

1. Score a baseline and a changed prompt with the same deterministic scorers
   used by release gates and production monitoring.
2. Apply a release gate with absolute thresholds and regression limits.
3. Record the outcome as a strict `DecisionRecord` — an `adopt` that cites a
   failing gate is rejected at the contract.
4. Watch `PromptManager.promote()` refuse to move the `production` alias
   without adopt-grade evidence persisted in a governed decision run.

The deterministic cells run with zero credentials. The final cell persists the
decision run and moves the alias against a connected workspace and stays off
by default. The practice-to-machinery map for all of this is the
[LLMOps playbook](../docs/llmops-playbook.md).

## 1. Score the baseline and the change with shared scorers

The fictional Aster Ridge earnings-summary assistant compares two exact prompt
templates: the baseline summarizes supplied facts, and the change adds exactly
one requirement — cite the supplied `source_id` once. A deterministic stand-in
application derives every answer from the exact template text, so the scores
measure the template under evaluation rather than hard-coded outputs.
`aai_core.scorers` holds the deterministic scorer definitions, so this
notebook, the CI release gate, and production monitoring cannot drift apart.
All figures are synthetic.

In [ ]:
from aai_core.scorers import score_all

BASELINE_TEMPLATE = (
    "Summarize only facts stated in {{earnings_excerpt}} for {{question}}."
)
CHANGE_TEMPLATE = (
    "Summarize only facts stated in {{earnings_excerpt}} for {{question}}. "
    "Cite {{source_id}} exactly once."
)

EVAL_CASES = [
    {
        "case_id": "quarterly-results",
        "source_id": "ARS-FY25-Q2-RESULTS",
        "expectations": {
            "expected_response": (
                "Fictional quarterly revenue reached $128.4 million with a "
                "12.1 percent operating margin [source: ARS-FY25-Q2-RESULTS]."
            )
        },
    },
    {
        "case_id": "forward-guidance",
        "source_id": "ARS-FY25-Q3-GUIDE",
        "expectations": {
            "expected_response": (
                "Fictional guidance projects $134 million revenue and a "
                "12.8 percent margin [source: ARS-FY25-Q3-GUIDE]."
            )
        },
    },
    {
        "case_id": "cash-and-risk",
        "source_id": "ARS-FY25-Q2-CASH-RISK",
        "expectations": {
            "expected_response": (
                "Fictional free cash flow was $21.7 million while inventory "
                "grew 18 percent [source: ARS-FY25-Q2-CASH-RISK]."
            )
        },
    },
]

FACTS = {
    "quarterly-results": (
        "Fictional quarterly revenue reached $128.4 million with a "
        "12.1 percent operating margin."
    ),
    "forward-guidance": (
        "Fictional guidance projects $134 million revenue and a "
        "12.8 percent margin."
    ),
    "cash-and-risk": (
        "Fictional free cash flow was $21.7 million while inventory "
        "grew 18 percent."
    ),
}


def run_application(template, case):
    """Deterministic stand-in for the assistant under test: it summarizes
    the case facts and obeys the template's citation instruction, so every
    answer derives from the exact template being evaluated."""
    answer = FACTS[case["case_id"]]
    if "{{source_id}}" in template:
        answer = f"{answer[:-1]} [source: {case['source_id']}]."
    return answer


def apply_template(template):
    return {
        case["case_id"]: run_application(template, case) for case in EVAL_CASES
    }


BASELINE_ANSWERS = apply_template(BASELINE_TEMPLATE)
CHANGE_ANSWERS = apply_template(CHANGE_TEMPLATE)


def citation_rate(answers):
    cited = sum(
        1
        for case in EVAL_CASES
        if answers[case["case_id"]].count(case["source_id"]) == 1
    )
    return cited / len(EVAL_CASES)


def mean_metrics(answers):
    totals = {}
    for case in EVAL_CASES:
        for name, value in score_all(
            answers[case["case_id"]], case["expectations"]
        ).items():
            totals.setdefault(name, []).append(value)
    metrics = {name: sum(values) / len(values) for name, values in totals.items()}
    metrics["citation_rate"] = citation_rate(answers)
    return metrics


baseline_metrics = mean_metrics(BASELINE_ANSWERS)
change_metrics = mean_metrics(CHANGE_ANSWERS)
for name in sorted(change_metrics):
    print(f"{name}: baseline={baseline_metrics[name]:.2f} "
          f"change={change_metrics[name]:.2f}")

## 2. Apply the deterministic release gate

`GatePolicy` states the release requirements once: the citation requirement is
absolute, and quality must not regress against the accepted baseline by more
than the stated budget. The gate consumes plain metric mappings here; against
a connected workspace the same `apply_gate()` consumes the native
`mlflow.genai.evaluate()` result unchanged.

In [ ]:
from aai_core.evaluation import (
    GatePolicy,
    MetricDirection,
    MetricRule,
    apply_gate,
)

gate_policy = GatePolicy(
    rules=(
        MetricRule(
            metric="citation_rate",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
        MetricRule(
            metric="keyword_coverage",
            direction=MetricDirection.HIGHER,
            required=0.8,
            max_regression=0.05,
        ),
        MetricRule(
            metric="response_length_ok",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
    )
)

change_gate = apply_gate(
    change_metrics, policy=gate_policy, baseline_metrics=baseline_metrics
)
print(f"change gate passed: {change_gate.passed}")

failing_gate = apply_gate(
    baseline_metrics, policy=gate_policy, baseline_metrics=baseline_metrics
)
for failure in failing_gate.failures:
    print(f"baseline would fail on {failure.metric}: {failure.reason}")

## 3. Record an explicit decision

A decision is persisted evidence, not a comment. `DecisionRecord` binds the
vocabulary to what it was decided from: the baseline and change runs, the gate
result, the qualified prompt identity, the exact template content digest, and
the release digest — the digest binds content while the name binds the
registry resource, so evidence for one prompt can never promote another that
happens to share a template. The contract refuses evidence-free adoption —
an `adopt` must cite a passing gate, never a failing or absent one — and
refuses personal identity in `decided_by`.

In [ ]:
from pydantic import ValidationError

from aai_core.decisions import Decision, DecisionRecord
from aai_core.prompts import prompt_digest

try:
    DecisionRecord(
        decision=Decision.ADOPT,
        change_id="earnings-summary-prompt-v2",
        change_summary="Require one exact source citation.",
        rationale="This adopt contradicts its own gate evidence.",
        gate=failing_gate,
    )
except ValidationError as error:
    print(f"contradiction refused: {error.errors()[0]['msg']}")

adopt_record = DecisionRecord(
    decision=Decision.ADOPT,
    change_id="earnings-summary-prompt-v2",
    change_summary="Require one exact source citation.",
    rationale=(
        "Citation rate reached 1.0 on every case with no keyword-coverage "
        "regression against the accepted baseline."
    ),
    gate=change_gate,
    prompt_name="main.app.earnings_summary",
    prompt_version=2,
    prompt_digest=prompt_digest(CHANGE_TEMPLATE),
    decided_by="group:app-owners",
)
# The connected cell below binds baseline_run_id/change_run_id to real runs.
print("searchable run tags:", adopt_record.as_tags())

## 4. Promotion refuses to move without adopt-grade, version-bound evidence

Aliases are deployment pointers, never release evidence. `promote()` requires
an `adopt` decision whose `prompt_digest` was recorded at decision time, *and*
the `decision_run_id` of the run that persisted it — in-memory evidence alone
never moves an alias, because a record that was never persisted leaves nothing
for an auditor to read. The two refusals below run with zero credentials and
never reach that persisted-run check: local evidence validation comes first, so
a `reject`/`inconclusive` decision is refused outright, and a bare gate is
refused because gate evidence alone carries no template identity (a digest
supplied at promotion time would prove only what is being promoted, not
what was evaluated). At promotion time the registry version's actual
template is digested and compared against the decision's digest, so
evidence gathered for one template can never move the alias onto another
version.

In [ ]:
from aai_core.decisions import Decision, DecisionRecord
from aai_core.prompts import PromptManager, PromptPromotionError, prompt_digest
from aai_core.testing import dev_settings

prompts = PromptManager(
    context=dev_settings().resource, catalog="main", schema="app"
)
print("change template digest:", prompt_digest(CHANGE_TEMPLATE)[:16], "...")

reject_record = DecisionRecord(
    decision=Decision.REJECT,
    change_id="earnings-summary-prompt-v2",
    change_summary="Require one exact source citation.",
    rationale="The gate failed the absolute citation requirement.",
    gate=failing_gate,
)
try:
    prompts.promote("earnings_summary", version=2, evidence=reject_record)
except PromptPromotionError as error:
    print(f"[{error.code}] {error}")

# A passing gate alone is not enough: it names no template content. Record
# the adopt decision (with its prompt_digest) and promote with that record.
try:
    prompts.promote("earnings_summary", version=2, evidence=change_gate)
except PromptPromotionError as error:
    print(f"[{error.code}] {error}")

## 5. Persist the decision and move the alias (connected)

Flip the flag after completing the connected setup lab
(`05_connected_setup.ipynb`). The cell registers the changed template
idempotently by content digest, records governed baseline and change runs
in the workspace so the persisted decision cites real run IDs, writes the
decision as a governed run with searchable `aai.decision` tags and a
`decision.json` artifact, and only then moves the `production` alias.
Rerunning it never mints duplicate prompt versions.

The change evidence is recomputed here from the registered artifact itself:
the stand-in application re-runs against the registry version's template and
the release gate is re-applied to those metrics, so an unexpected registered
template fails the gate instead of being promoted. The regression baseline
is likewise whatever `production` currently serves — the adopted version is
loaded from the registry and evaluated through the same stand-in
application, so the proposed change can never pass by comparing against a stale
constant; the hard-coded baseline template only seeds the very first
adoption, before any alias exists. The decision binds the qualified prompt
name, the registered version, and the content digest, and `promote()`
re-verifies all three against the registry, so the evidence can never
describe different content — or a different prompt — than the alias moves
to. A production project produces the same evidence shape from
the registered version through the governed release gate instead
(`evaluate_with_gate` inside the templates' `release_gate` job).

Graduation map: `prompt-app` templates carry the register → evaluate →
promote shape as `scripts/register_prompt.py`, `evals/evaluate.py`, and
`scripts/promote_prompt.py`. Today `promote_prompt.py` still moves the
alias with `set_alias()` and documents that the release gate must have
passed for exactly that version; routing it through `promote()` with a
recorded `DecisionRecord` lands with each template's next release, which
is when templates pick up the aai-core version shipping this API. The
decision run makes the outcome searchable next to the evaluation
evidence.

In [ ]:
PERSIST_EVIDENCE_TO_DATABRICKS = False

if PERSIST_EVIDENCE_TO_DATABRICKS:
    from aai_core import bootstrap
    from aai_core.decisions import record_decision
    from aai_core.evaluation import log_gate_evidence
    from aai_core.experiments import ExperimentRunMetadata, RunPurpose

    context = bootstrap()
    registered = context.prompts.ensure_version(
        "earnings_summary",
        CHANGE_TEMPLATE,
        commit_message="Require one exact source citation",
        tags={"experiment_role": "change"},
    )
    mlflow = context.experiments.native_client

    # The regression baseline is whatever production currently serves:
    # load the adopted version and evaluate it through the same stand-in
    # application. Only a genuinely absent alias may seed the first
    # adoption from the hard-coded baseline; authentication, permission,
    # and transient registry failures propagate, or the change would be
    # gated against the wrong baseline.
    from aai_core.prompts import is_missing_prompt_error

    try:
        # Release-critical lookup: bypass the 60-second alias cache so a
        # rerun never gates against a version production no longer serves.
        accepted = context.prompts.load(
            "earnings_summary", alias="production", cache_ttl_seconds=0
        )
        accepted_template = accepted.template
        accepted_label = f"production v{accepted.version}"
    except Exception as error:
        if not is_missing_prompt_error(error):
            raise
        accepted_template = BASELINE_TEMPLATE
        accepted_label = "first adoption (no production alias yet)"
    accepted_metrics = mean_metrics(apply_template(accepted_template))
    print(f"regression baseline: {accepted_label}")

    # The persisted evidence derives from the registered artifact: re-run
    # the stand-in application against the registry version's template and
    # re-apply the release gate before recording anything.
    registered_metrics = mean_metrics(apply_template(registered.template))
    registered_gate = apply_gate(
        registered_metrics,
        policy=gate_policy,
        baseline_metrics=accepted_metrics,
    )
    if not registered_gate.passed:
        raise RuntimeError(
            "registered template failed the release gate; refusing to "
            "record adopt evidence"
        )

    with context.experiments.run(
        run_name="baseline-earnings-summary-accepted",
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.BASELINE,
            change_id="earnings-summary-prompt-v2",
            change_summary="Require one exact source citation.",
        ),
    ) as baseline_run:
        mlflow.log_metrics(accepted_metrics)
        baseline_run_id = baseline_run.info.run_id

    with context.experiments.run(
        run_name="change-cited-earnings-summary-prompt-v2",
        metadata=ExperimentRunMetadata(
            purpose=RunPurpose.CHANGE,
            change_id="earnings-summary-prompt-v2",
            change_summary="Require one exact source citation.",
            baseline_run_id=baseline_run_id,
        ),
    ) as change_run:
        mlflow.log_metrics(registered_metrics)
        log_gate_evidence(registered_gate)
        change_run_id = change_run.info.run_id

    persisted_record = DecisionRecord(
        decision=Decision.ADOPT,
        change_id="earnings-summary-prompt-v2",
        change_summary="Require one exact source citation.",
        rationale=adopt_record.rationale,
        baseline_run_id=baseline_run_id,
        change_run_id=change_run_id,
        gate=registered_gate,
        prompt_name=context.prompts.qualify("earnings_summary"),
        prompt_version=registered.version,
        prompt_digest=prompt_digest(registered.template),
        decided_by="group:app-owners",
    )
    decision_run_id = record_decision(
        persisted_record, experiments=context.experiments
    )
    # promote() verifies the persisted run, so the decision_run_id from
    # record_decision() above is required; the in-memory record is the
    # optional cross-check that must match what was persisted.
    context.prompts.promote(
        "earnings_summary",
        version=registered.version,
        decision_run_id=decision_run_id,
        evidence=persisted_record,
    )
    print(
        f"baseline {baseline_run_id} and change {change_run_id} recorded; "
        f"decision run {decision_run_id} moved the production alias to "
        f"version {registered.version}"
    )